# MCP 클라이언트 구현

**Skilljar Lesson 06 대응**

이 노트북에서 다루는 내용:
1. MCP 클라이언트로 서버에 연결
2. 도구 목록 조회 및 호출
3. Claude API와 MCP 클라이언트 통합
4. Week 04 Tool Use 루프 패턴 재활용

In [ ]:
# ── Setup ──────────────────────────────────────────────
import asyncio
import json
import anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()

MODEL = "claude-haiku-4-5"

## §1. MCP 클라이언트 기본 연결

MCP 클라이언트는 서버에 연결하여 도구를 조회하고 호출합니다.

1. `StdioServerParameters`로 서버 연결 정보 설정
2. `stdio_client`로 서버 프로세스 시작
3. `ClientSession`으로 세션 생성
4. `session.initialize()`로 핸드셰이크
5. `session.list_tools()` / `session.call_tool()`로 도구 사용

In [ ]:
async def connect_and_list_tools():
    """MCP 서버에 연결하여 도구 목록을 조회합니다."""

    # 서버 연결 파라미터 (server.py를 프로세스로 실행)
    server_params = StdioServerParameters(
        command="python",
        args=["server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 세션 초기화 (핸드셰이크)
            await session.initialize()
            print("서버에 연결되었습니다.\n")

            # 도구 목록 조회
            tools = await session.list_tools()
            print(f"사용 가능한 도구 ({len(tools.tools)}개):")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

            return tools

tools = await connect_and_list_tools()

## §2. 도구 호출 (call_tool)

서버에 연결한 상태에서 `session.call_tool()`로 도구를 호출합니다.

In [ ]:
async def call_tools_demo():
    """서버의 도구들을 호출하여 테스트합니다."""

    server_params = StdioServerParameters(
        command="python", args=["server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 도구 1: 현재 시간
            result1 = await session.call_tool(
                "get_current_time",
                arguments={"format": "%Y-%m-%d %H:%M"}
            )
            print(f"현재 시간: {result1.content[0].text}")

            # 도구 2: 덧셈
            result2 = await session.call_tool(
                "add_numbers",
                arguments={"a": 3.14, "b": 2.71}
            )
            print(f"3.14 + 2.71 = {result2.content[0].text}")

            # 도구 3: 면적 계산
            result3 = await session.call_tool(
                "calculate_area",
                arguments={"width": 300, "height": 600, "unit": "mm"}
            )
            print(f"면적: {result3.content[0].text}")

await call_tools_demo()

## §3. Claude API와 MCP 클라이언트 통합

MCP 클라이언트의 진정한 가치: **Claude API와 결합**하여 Claude가 MCP 도구를 자율적으로 사용하게 합니다.

핵심 아이디어:
1. MCP 서버의 도구 목록을 **Claude Tool Use 형식으로 변환**
2. Week 04의 **`stop_reason` 기반 루프**를 그대로 활용
3. 도구 실행만 **MCP 클라이언트를 통해 서버에 위임**

In [ ]:
async def chat_with_mcp_tools(user_message: str):
    """MCP 서버의 도구를 사용하여 Claude와 대화합니다."""

    api_client = anthropic.Anthropic()
    server_params = StdioServerParameters(
        command="python", args=["server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Step 1: MCP 도구 → Claude Tool Use 형식 변환
            mcp_tools = await session.list_tools()
            claude_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                }
                for tool in mcp_tools.tools
            ]

            messages = [{"role": "user", "content": user_message}]
            print(f"👤 사용자: {user_message}\n")

            # Step 2: Week 04 패턴과 동일한 Tool Use 루프
            while True:
                response = api_client.messages.create(
                    model=MODEL,
                    max_tokens=1024,
                    tools=claude_tools,
                    messages=messages
                )

                if response.stop_reason == "end_turn":
                    # 최종 응답
                    final_text = "".join(
                        block.text for block in response.content
                        if hasattr(block, "text")
                    )
                    print(f"🤖 Claude: {final_text}")
                    return final_text

                elif response.stop_reason == "tool_use":
                    # Claude가 도구 호출 요청
                    messages.append({
                        "role": "assistant",
                        "content": response.content
                    })

                    tool_results = []
                    for block in response.content:
                        if block.type == "tool_use":
                            print(f"  🔧 MCP 도구 호출: {block.name}({block.input})")

                            # Step 3: MCP 서버에서 도구 실행!
                            result = await session.call_tool(
                                block.name,
                                arguments=block.input
                            )
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": result.content[0].text
                            })

                    messages.append({
                        "role": "user",
                        "content": tool_results
                    })

In [ ]:
# 테스트: Claude가 MCP 도구를 자율적으로 사용
await chat_with_mcp_tools("지금 몇 시야?")

In [ ]:
# 테스트: 면적 계산 요청
await chat_with_mcp_tools("가로 300mm, 세로 600mm인 직사각형의 면적을 계산해줘")

## 핵심 정리

| 단계 | Week 04 (Tool Use) | Week 07 (MCP) |
|------|-------------------|---------------|
| 도구 목록 | `tools` 배열에 스키마 직접 포함 | `session.list_tools()` → 자동 변환 |
| 도구 실행 | `run_tool()` 함수에서 직접 실행 | `session.call_tool()` → 서버에 위임 |
| 루프 패턴 | `stop_reason == "tool_use"` | 동일! |
| 결과 전달 | `tool_result` 직접 구성 | 동일! |

**핵심**: Tool Use 루프 패턴은 그대로, 도구 실행만 MCP 서버로 위임!